# Physics hourly yield (fallback for composite S)

Hourly physics integration on **NY gateway cells only** (~1,089) for yield validation and optional P50 scoring.

| Input | Output |
|-------|--------|
| `data/raw_solar_wind_hourly_data/NY_State_10km_YYYY.csv` | `data/processed/cell_physics_yield.parquet` |
| `cells_passing_gateway.parquet` (`in_ny_state`) | `physics_validation_report.json` |

**Solar:** Simplified GHI → AC (no PVLib dependency): `CF_h = (GHI/1000) × PR × temp_derate` daytime, 0 at night → annual mean. Temp derate: `1 − 0.004×(T−25°C)`.

**Wind:** Generic 2.5 MW curve on 80 m hub speed (α=0.14 shear); cut-in 3 / rated 12 / cut-out 25 m/s.

**P50 / P90:** Median and 5th percentile of **year-level** physics CF across 2018–2025.

**Requires:** `01` (raw hourly CSVs), `06` (gateway), `03` (EIA validation)

> **Tip:** Set `YEARS = [2025]` for a quick test (~10 s).

In [1]:
# --- User settings ---
YEARS = None  # None = all years in config
USE_SCRATCH_CACHE = True
INVALIDATE_SCRATCH = False  # set True after changing physics formulas

import json
import shutil
import warnings
warnings.filterwarnings('ignore')

from pathlib import Path

import polars as pl
import yaml

ROOT = Path('../..').resolve()
CONFIG_PATH = ROOT / 'config' / 'paths.yaml'
GATEWAY_PATH = ROOT / 'data' / 'processed' / 'cells_passing_gateway.parquet'
EIA_PATH = ROOT / 'data' / 'processed' / 'eia_facility_features.parquet'
OUTPUT_PATH = ROOT / 'data' / 'processed' / 'cell_physics_yield.parquet'
REPORT_PATH = ROOT / 'data' / 'processed' / 'physics_validation_report.json'
SCRATCH_DIR = ROOT / 'data' / 'processed' / 'scratch' / 'physics_cell_year'

P90_PERCENTILE = 0.05
HUB_HEIGHT_M = 80.0
WIND_SHEAR_ALPHA = 0.14
HUB_SPEED_FACTOR = (HUB_HEIGHT_M / 10.0) ** WIND_SHEAR_ALPHA
SOLAR_PERFORMANCE_RATIO = 0.82
WIND_CUT_IN_MPS = 3.0
WIND_RATED_MPS = 12.0
WIND_CUT_OUT_MPS = 25.0

with CONFIG_PATH.open(encoding='utf-8') as f:
    CONFIG = yaml.safe_load(f)

GRID_DIR = ROOT / CONFIG['data']['raw']['grid_weather_dir']
YEARS = YEARS or CONFIG['weather']['grid_years']

SCRATCH_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

if INVALIDATE_SCRATCH and SCRATCH_DIR.exists():
    shutil.rmtree(SCRATCH_DIR)
    SCRATCH_DIR.mkdir(parents=True, exist_ok=True)
    print('Invalidated physics scratch:', SCRATCH_DIR)

print('ROOT:', ROOT)
print('Years:', YEARS)
print('Hub speed factor:', round(HUB_SPEED_FACTOR, 4))

Invalidated physics scratch: /anvil/projects/x-eve260007/Adaptive-Machine-Learning-for-Clean-Energy-Investment-in-New-York/data/processed/scratch/physics_cell_year
ROOT: /anvil/projects/x-eve260007/Adaptive-Machine-Learning-for-Clean-Energy-Investment-in-New-York
Years: [2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025]
Hub speed factor: 1.3379


In [2]:
def wind_power_fraction(v_hub: pl.Expr) -> pl.Expr:
    """Normalized generic 2.5 MW power curve (0–1) on hub-height speed."""
    cubic = (
        (v_hub.pow(3) - WIND_CUT_IN_MPS ** 3)
        / (WIND_RATED_MPS ** 3 - WIND_CUT_IN_MPS ** 3)
    ).clip(0, 1)
    return (
        pl.when(v_hub < WIND_CUT_IN_MPS).then(0.0)
        .when(v_hub >= WIND_CUT_OUT_MPS).then(0.0)
        .when(v_hub >= WIND_RATED_MPS).then(1.0)
        .otherwise(cubic)
    )


def physics_cf_for_year(csv_path: Path, year: int, cells: pl.DataFrame) -> pl.DataFrame:
    """Filter hourly CSV to target cells; return one row per cell with annual physics CF."""
    hourly = (
        pl.scan_csv(csv_path, ignore_errors=True)
        .filter(pl.col('latitude').is_not_null() & pl.col('longitude').is_not_null())
        .join(cells.lazy(), on=['latitude', 'longitude'], how='semi')
        .with_columns(
            pl.col('shortwave_radiation').clip(lower_bound=0).alias('ghi'),
            (pl.col('wind_speed_10m') * HUB_SPEED_FACTOR).alias('v_hub'),
            (1.0 - 0.004 * (pl.col('temperature_2m') - 25.0))
            .clip(0.75, 1.05)
            .alias('temp_derate'),
        )
        .with_columns(
            pl.when(pl.col('ghi') > 0)
            .then(pl.col('ghi') / 1000.0 * SOLAR_PERFORMANCE_RATIO * pl.col('temp_derate'))
            .otherwise(0.0)
            .alias('solar_frac'),
            wind_power_fraction(pl.col('v_hub')).alias('wind_frac'),
        )
        .group_by(['latitude', 'longitude'])
        .agg(
            pl.col('solar_frac').mean().alias('physics_solar_cf'),
            pl.col('wind_frac').mean().alias('physics_wind_cf'),
        )
        .with_columns(
            pl.lit(year).alias('year'),
            (
                (pl.col('physics_solar_cf') + pl.col('physics_wind_cf')) / 2.0
            ).alias('physics_hybrid_cf'),
        )
        .collect()
    )
    return hourly


print('Physics helpers ready')

Physics helpers ready


In [3]:
cells = (
    pl.read_parquet(GATEWAY_PATH)
    .filter(pl.col('in_ny_state'))
    .select(['latitude', 'longitude'])
    .unique()
)
print(f'Target NY gateway cells: {cells.height:,}')

year_frames: list[pl.DataFrame] = []
for year in YEARS:
    scratch_file = SCRATCH_DIR / f'physics_cf_{year}.parquet'
    if USE_SCRATCH_CACHE and scratch_file.exists():
        year_df = pl.read_parquet(scratch_file)
        print(f'  {year}: cached ({year_df.height:,} cells)')
    else:
        csv_path = GRID_DIR / f'NY_State_10km_{year}.csv'
        if not csv_path.exists():
            raise FileNotFoundError(f'Missing hourly CSV: {csv_path} (run 01 / 00 first)')
        year_df = physics_cf_for_year(csv_path, year, cells)
        year_df.write_parquet(scratch_file)
        print(f'  {year}: computed ({year_df.height:,} cells) -> {scratch_file.name}')
    year_frames.append(year_df)

yearly = pl.concat(year_frames)
print(f'Cell-years: {yearly.height:,}')

Target NY gateway cells: 1,089


  2018: computed (965 cells) -> physics_cf_2018.parquet


  2019: computed (916 cells) -> physics_cf_2019.parquet


  2020: computed (917 cells) -> physics_cf_2020.parquet


  2021: computed (1,089 cells) -> physics_cf_2021.parquet


  2022: computed (919 cells) -> physics_cf_2022.parquet


  2023: computed (1,089 cells) -> physics_cf_2023.parquet


  2024: computed (916 cells) -> physics_cf_2024.parquet


  2025: computed (913 cells) -> physics_cf_2025.parquet
Cell-years: 7,724


In [4]:
def aggregate_physics(yearly: pl.DataFrame) -> pl.DataFrame:
    return yearly.group_by(['latitude', 'longitude']).agg(
        pl.col('physics_solar_cf').median().alias('physics_solar_cf_p50'),
        pl.col('physics_wind_cf').median().alias('physics_wind_cf_p50'),
        pl.col('physics_hybrid_cf').median().alias('physics_hybrid_cf_p50'),
        pl.col('physics_solar_cf').quantile(P90_PERCENTILE).alias('physics_solar_cf_p90'),
        pl.col('physics_wind_cf').quantile(P90_PERCENTILE).alias('physics_wind_cf_p90'),
        pl.col('physics_hybrid_cf').quantile(P90_PERCENTILE).alias('physics_hybrid_cf_p90'),
        pl.col('year').n_unique().alias('physics_n_years'),
    )


physics = aggregate_physics(yearly)
physics.write_parquet(OUTPUT_PATH)
print(f'Wrote {physics.height:,} cells -> {OUTPUT_PATH}')
physics.select([
    'physics_solar_cf_p50', 'physics_wind_cf_p50', 'physics_hybrid_cf_p50',
]).describe()

Wrote 1,089 cells -> /anvil/projects/x-eve260007/Adaptive-Machine-Learning-for-Clean-Energy-Investment-in-New-York/data/processed/cell_physics_yield.parquet


statistic,physics_solar_cf_p50,physics_wind_cf_p50,physics_hybrid_cf_p50
str,f64,f64,f64
"""count""",1089.0,1089.0,1089.0
"""null_count""",0.0,0.0,0.0
"""mean""",0.133369,0.073414,0.103344
"""std""",0.002813,0.023144,0.011779
"""min""",0.125798,0.019189,0.077486
"""25%""",0.132122,0.05822,0.095614
"""50%""",0.133595,0.071175,0.102228
"""75%""",0.134792,0.087009,0.110241
"""max""",0.140848,0.224275,0.18091


In [5]:
import numpy as np


def oof_r2(y_true, y_pred) -> float:
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    mask = np.isfinite(y_true) & np.isfinite(y_pred)
    if mask.sum() < 3:
        return float('nan')
    yt, yp = y_true[mask], y_pred[mask]
    ss_res = ((yt - yp) ** 2).sum()
    ss_tot = ((yt - yt.mean()) ** 2).sum()
    if ss_tot == 0:
        return float('nan')
    return float(1 - ss_res / ss_tot)


def pearson_r(y_true, y_pred) -> float:
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    mask = np.isfinite(y_true) & np.isfinite(y_pred)
    if mask.sum() < 3:
        return float('nan')
    return float(np.corrcoef(y_true[mask], y_pred[mask])[0, 1])


VALIDATION_YEARS = [y for y in range(2018, 2024) if y in YEARS]

eia = pl.read_parquet(EIA_PATH).filter(
    pl.col('cf_valid_flag')
    & pl.col('snap_within_5km')
    & pl.col('year').is_in(VALIDATION_YEARS)
)

# Cell-year validation (physics is cell-based; aggregate EIA plants snapped to same cell)
solar_cell_year = (
    eia.filter(pl.col('technology') == 'solar')
    .group_by(['grid_latitude', 'grid_longitude', 'year'])
    .agg(pl.col('capacity_factor').mean().alias('eia_cf'))
    .rename({'grid_latitude': 'latitude', 'grid_longitude': 'longitude'})
    .join(yearly, on=['latitude', 'longitude', 'year'], how='inner')
)
wind_cell_year = (
    eia.filter(pl.col('technology') == 'wind')
    .group_by(['grid_latitude', 'grid_longitude', 'year'])
    .agg(pl.col('capacity_factor').mean().alias('eia_cf'))
    .rename({'grid_latitude': 'latitude', 'grid_longitude': 'longitude'})
    .join(yearly, on=['latitude', 'longitude', 'year'], how='inner')
)

solar_r2 = oof_r2(solar_cell_year['eia_cf'], solar_cell_year['physics_solar_cf'])
solar_r = pearson_r(solar_cell_year['eia_cf'], solar_cell_year['physics_solar_cf'])
wind_r2 = oof_r2(wind_cell_year['eia_cf'], wind_cell_year['physics_wind_cf'])
wind_r = pearson_r(wind_cell_year['eia_cf'], wind_cell_year['physics_wind_cf'])

report = {
    'method': 'simplified_hourly_physics',
    'solar_model': 'ghi_pr_temp_derate_no_pvlib',
    'wind_model': 'generic_2p5mw_power_curve_hub80m',
    'n_ny_cells': int(physics.height),
    'n_years': len(YEARS),
    'validation_years': VALIDATION_YEARS,
    'p90_percentile': P90_PERCENTILE,
    'validation_unit': 'cell_year_mean_eia',
    'solar_vs_eia_r2': solar_r2,
    'solar_vs_eia_pearson_r': solar_r,
    'solar_vs_eia_n_cell_years': int(solar_cell_year.height),
    'wind_vs_eia_r2': wind_r2,
    'wind_vs_eia_pearson_r': wind_r,
    'wind_vs_eia_n_cell_years': int(wind_cell_year.height),
    'r6_solar_acceptance_target_r2': 0.1,
    'solar_acceptance_pass': bool(solar_r2 > 0.1) if np.isfinite(solar_r2) else False,
    'note': (
        'Physics solar CF is resource-grade (~13%); EIA operational CF ~3.5%. '
        'Raw R² is scale-dominated; Pearson r captures spatial agreement at cell-year level.'
    ),
    'wind_limitation': 'Small wind sample (~31 plants); physics curve is screening-grade only',
}
REPORT_PATH.write_text(json.dumps(report, indent=2) + '\n')

print(
    f'Solar cell-year vs EIA: R²={solar_r2:.3f}, r={solar_r:.3f} '
    f'(n={solar_cell_year.height}, target R² > 0.1)'
)
print(
    f'Wind cell-year vs EIA: R²={wind_r2:.3f}, r={wind_r:.3f} '
    f'(n={wind_cell_year.height})'
)
print(f'Report -> {REPORT_PATH}')

Solar cell-year vs EIA: R²=-35.243, r=0.286 (n=499, target R² > 0.1)
Wind cell-year vs EIA: R²=-17.037, r=-0.093 (n=95)
Report -> /anvil/projects/x-eve260007/Adaptive-Machine-Learning-for-Clean-Energy-Investment-in-New-York/data/processed/physics_validation_report.json
